<h1>Table of Contents<span class="tocSkip"></span></h1>
<div class="toc"><ul class="toc-item"><li><span><a href="#Row" data-toc-modified-id="Row-1"><span class="toc-item-num">1&nbsp;&nbsp;</span>Row</a></span></li><li><span><a href="#Colonnes" data-toc-modified-id="Colonnes-2"><span class="toc-item-num">2&nbsp;&nbsp;</span>Colonnes</a></span></li><li><span><a href="#expr" data-toc-modified-id="expr-3"><span class="toc-item-num">3&nbsp;&nbsp;</span>expr</a></span></li><li><span><a href="#Transformations" data-toc-modified-id="Transformations-4"><span class="toc-item-num">4&nbsp;&nbsp;</span>Transformations</a></span><ul class="toc-item"><li><span><a href="#Projections" data-toc-modified-id="Projections-4.1"><span class="toc-item-num">4.1&nbsp;&nbsp;</span>Projections</a></span><ul class="toc-item"><li><span><a href="#selectExpr" data-toc-modified-id="selectExpr-4.1.1"><span class="toc-item-num">4.1.1&nbsp;&nbsp;</span>selectExpr</a></span></li><li><span><a href="#drop" data-toc-modified-id="drop-4.1.2"><span class="toc-item-num">4.1.2&nbsp;&nbsp;</span>drop</a></span></li><li><span><a href="#orderBy" data-toc-modified-id="orderBy-4.1.3"><span class="toc-item-num">4.1.3&nbsp;&nbsp;</span>orderBy</a></span></li></ul></li></ul></li></ul></div>

In [1]:
from pyspark.sql import SparkSession
from pyspark.sql.functions import *
from pyspark.sql.types     import StructType, \
     StructField, FloatType, \
     IntegerType, StringType
import os, warnings

warnings.filterwarnings(action="ignore")

In [2]:
spark = (SparkSession.builder
         .appName("01-API.DataFrames-colonnes-projections")
         .getOrCreate())

print("Master :", spark.sparkContext.master)
print("Application :", spark.sparkContext.applicationId)
print("Python :", os.sys.version.split()[0])
print("Spark :", spark.version)

26/09/24 09:49:20 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable
26/09/24 09:49:21 WARN MetricsConfig: Cannot locate configuration: tried hadoop-metrics2-s3a-file-system.properties,hadoop-metrics2.properties
SLF4J: Failed to load class "org.slf4j.impl.StaticLoggerBinder".
SLF4J: Defaulting to no-operation (NOP) logger implementation
SLF4J: See http://www.slf4j.org/codes.html#StaticLoggerBinder for further details.
26/09/24 09:49:22 WARN S3ABlockOutputStream: Application invoked the Syncable API against stream writing to spark-events/eventlog_v2_app-20260924094921-0012/events_1_app-20260924094921-0012.zstd. This is Unsupported


Master : spark://spark-master:7077
Application : app-20260924094921-0012
Python : 3.10.12
Spark : 4.0.4


In [3]:
spark

# Row
Un DataFrame est un objet de type **« Dataset[Row] »**, ainsi toutes les caractéristiques de l’objet **« Dataset »** sont également valables pour les **DataFrames**.

In [4]:
from pyspark.sql import Row
from pyspark.sql.functions import *

uneLigne = Row(1,"Isabelle","BIZOÏ")
uneAutre = Row(2,"Razvan","BIZOÏ")
listeLignes = [uneLigne,uneAutre]
personnes = spark.createDataFrame(listeLignes,["Id","Prenom","Nom"])
personnes.show()

[Stage 1:=======================================>                   (2 + 1) / 3]

+---+--------+-----+
| Id|  Prenom|  Nom|
+---+--------+-----+
|  1|Isabelle|BIZOÏ|
|  2|  Razvan|BIZOÏ|
+---+--------+-----+



# Colonnes

<img src="https://raw.githubusercontent.com/rbizoi/AnalyserLesDonneesAvecSpark/main/DataFrameSpark/images/M06-01.png" width="400">    

In [5]:
!ls -al ../data/meteo

total 143604
drwxrwxrwx 1 root root     512 Sep 24 08:10 .
drwxrwxrwx 1 root root     512 Sep 24 09:43 ..
-rwxrwxrwx 1 root root 4189020 Jan 31  2023 synop.202301.csv
-rwxrwxrwx 1 root root 3626128 Feb 28  2023 synop.202302.csv
-rwxrwxrwx 1 root root 4105061 Mar 31  2023 synop.202303.csv
-rwxrwxrwx 1 root root 4041942 Apr 30  2023 synop.202304.csv
-rwxrwxrwx 1 root root 4203446 May 31  2023 synop.202305.csv
-rwxrwxrwx 1 root root 4083804 Jun 30  2023 synop.202306.csv
-rwxrwxrwx 1 root root 4098443 Jul 31  2023 synop.202307.csv
-rwxrwxrwx 1 root root 4215978 Aug 31  2023 synop.202308.csv
-rwxrwxrwx 1 root root 4054875 Sep 30  2023 synop.202309.csv
-rwxrwxrwx 1 root root 4107655 Oct 31  2023 synop.202310.csv
-rwxrwxrwx 1 root root 4009103 Nov 30  2023 synop.202311.csv
-rwxrwxrwx 1 root root 4186069 Dec 31  2023 synop.202312.csv
-rwxrwxrwx 1 root root 4164769 Jan 31  2024 synop.202401.csv
-rwxrwxrwx 1 root root 3865487 Feb 29  2024 synop.202402.csv
-rwxrwxrwx 1 root root 4229957 Mar 31  2

In [6]:
meteoDataFrame  = spark.read.format('csv')\
    .option('sep',';')\
    .option('header','true')\
    .option('nullValue','mq')\
    .option('inferSchema', 'true')\
    .load('../data/meteo/')\
    .cache()

26/09/24 09:49:29 WARN SparkStringUtils: Truncated the string representation of a plan since it was too large. This behavior can be adjusted by setting 'spark.sql.debug.maxToStringFields'.


In [7]:
meteoDataFrame.printSchema()

root
 |-- numer_sta: integer (nullable = true)
 |-- date: long (nullable = true)
 |-- pmer: integer (nullable = true)
 |-- tend: integer (nullable = true)
 |-- cod_tend: integer (nullable = true)
 |-- dd: integer (nullable = true)
 |-- ff: double (nullable = true)
 |-- t: double (nullable = true)
 |-- td: double (nullable = true)
 |-- u: integer (nullable = true)
 |-- vv: integer (nullable = true)
 |-- ww: integer (nullable = true)
 |-- w1: integer (nullable = true)
 |-- w2: integer (nullable = true)
 |-- n: integer (nullable = true)
 |-- nbas: integer (nullable = true)
 |-- hbas: integer (nullable = true)
 |-- cl: integer (nullable = true)
 |-- cm: integer (nullable = true)
 |-- ch: integer (nullable = true)
 |-- pres: integer (nullable = true)
 |-- niv_bar: integer (nullable = true)
 |-- geop: integer (nullable = true)
 |-- tend24: integer (nullable = true)
 |-- tn12: double (nullable = true)
 |-- tn24: double (nullable = true)
 |-- tx12: double (nullable = true)
 |-- tx24: double (n

In [8]:
len(meteoDataFrame.columns), meteoDataFrame.columns

(60,
 ['numer_sta',
  'date',
  'pmer',
  'tend',
  'cod_tend',
  'dd',
  'ff',
  't',
  'td',
  'u',
  'vv',
  'ww',
  'w1',
  'w2',
  'n',
  'nbas',
  'hbas',
  'cl',
  'cm',
  'ch',
  'pres',
  'niv_bar',
  'geop',
  'tend24',
  'tn12',
  'tn24',
  'tx12',
  'tx24',
  'tminsol',
  'sw',
  'tw',
  'raf10',
  'rafper',
  'per',
  'etat_sol',
  'ht_neige',
  'ssfrai',
  'perssfrai',
  'rr1',
  'rr3',
  'rr6',
  'rr12',
  'rr24',
  'phenspe1',
  'phenspe2',
  'phenspe3',
  'phenspe4',
  'nnuage1',
  'ctype1',
  'hnuage1',
  'nnuage2',
  'ctype2',
  'hnuage2',
  'nnuage3',
  'ctype3',
  'hnuage3',
  'nnuage4',
  'ctype4',
  'hnuage4',
  '_c59'])

In [9]:
meteoDataFrame.select('numer_sta',"date","t").show(3)

[Stage 5:>                                                          (0 + 1) / 1]

+---------+--------------+------+
|numer_sta|          date|     t|
+---------+--------------+------+
|     7005|20240501000000|286.25|
|     7015|20240501000000|287.65|
|     7020|20240501000000|281.95|
+---------+--------------+------+
only showing top 3 rows


In [10]:
meteoDataFrame.numer_sta,meteoDataFrame["numer_sta"],meteoDataFrame.t

(Column<'numer_sta'>, Column<'numer_sta'>, Column<'t'>)

# expr
La fonction permet d’écrire directement une expression qui est exécutée pour l’ensemble des lignes.

In [11]:
meteoDataFrame.select('numer_sta',
                      expr('t  - 273.15').alias('temperature'),
                      expr('(t + pres/100)*vv/100').alias('calc')
        ).show(3)

+---------+------------------+----------+
|numer_sta|       temperature|      calc|
+---------+------------------+----------+
|     7005|13.100000000000023|  182832.1|
|     7015|              14.5|207307.975|
|     7020| 8.800000000000011|      NULL|
+---------+------------------+----------+
only showing top 3 rows


# Transformations

<img src="https://raw.githubusercontent.com/rbizoi/AnalyserLesDonneesAvecSpark/main/DataFrameSpark/images/M06-02.png" width="400"> 

## Projections

In [12]:
from pyspark.sql.types     import StructType, \
     StructField, FloatType, \
     IntegerType, StringType

schema = StructType([
        StructField('Id'           , StringType() , True),
        StructField('ville'        , StringType() , True),
        StructField('latitude'     , FloatType() , True),
        StructField('longitude'    , FloatType() , True),
        StructField('altitude'     , IntegerType() , True)])

villes  = spark.read.format('csv')   \
      .option('sep',';')                \
      .option('mergeSchema', 'true')    \
      .option('header','true')          \
      .schema(schema)                   \
      .load('../data/postesSynop.csv')  \
      .cache()

@udf("string")
def formatVille(ville):
    if ville in ['CLERMONT-FD','MONT-DE-MARSAN',
                                   'ST-PIERRE','ST-BARTHELEMY METEO'] :
        return ville.title()
    else :
        if ville.find('-') != -1 :
            return ville[0:ville.find('-')].title()
        else:
            return ville.title()

villesT  = villes.select(
                col('Id').alias('id'),
                formatVille('ville').alias('ville'),
               'latitude',
               'longitude',
               'altitude')

### selectExpr

In [13]:
villes.selectExpr('*','altitude * 1000 as alt').show(3)

+-----+---------------+---------+---------+--------+-----+
|   Id|          ville| latitude|longitude|altitude|  alt|
+-----+---------------+---------+---------+--------+-----+
|07005|      ABBEVILLE|   50.136|    1.834|      69|69000|
|07015|  LILLE-LESQUIN|    50.57|   3.0975|      47|47000|
|07020|PTE DE LA HAGUE|49.725166|-1.939833|       6| 6000|
+-----+---------------+---------+---------+--------+-----+
only showing top 3 rows


In [14]:
villesT.selectExpr('*','altitude * 1000 as alt').show(3)

+-----+---------------+---------+---------+--------+-----+
|   id|          ville| latitude|longitude|altitude|  alt|
+-----+---------------+---------+---------+--------+-----+
|07005|      Abbeville|   50.136|    1.834|      69|69000|
|07015|          Lille|    50.57|   3.0975|      47|47000|
|07020|Pte De La Hague|49.725166|-1.939833|       6| 6000|
+-----+---------------+---------+---------+--------+-----+
only showing top 3 rows


In [15]:
villes.drop('Id', 'latitude', 'longitude').show(3)

+---------------+--------+
|          ville|altitude|
+---------------+--------+
|      ABBEVILLE|      69|
|  LILLE-LESQUIN|      47|
|PTE DE LA HAGUE|       6|
+---------------+--------+
only showing top 3 rows


### drop

In [16]:
villes.drop('Id', 'latitude', 'longitude')\
       .orderBy('ville').show(5)

+-------------+--------+
|        ville|altitude|
+-------------+--------+
|    ABBEVILLE|      69|
|      AJACCIO|       5|
|      ALENCON|     143|
|BALE-MULHOUSE|     263|
|       BASTIA|      10|
+-------------+--------+
only showing top 5 rows


In [17]:
villes.drop('Id', 'latitude', 'longitude')\
       .orderBy(desc('altitude')).show(5)

+------------------+--------+
|             ville|altitude|
+------------------+--------+
|            EMBRUN|     871|
|     LE PUY-LOUDES|     833|
|            MILLAU|     712|
|         ST GIRONS|     414|
|LIMOGES-BELLEGARDE|     402|
+------------------+--------+
only showing top 5 rows


In [18]:
meteo = meteoDataFrame.select(
                 col('numer_sta'),
                 col('date')[0:4].cast('int') ,
                 col('date')[5:2].cast('int'),
                 col('date')[7:2].cast('int'),
                 col('date')[5:4],
                 round(col('t') - 273.15,2),
                 col('u') / 100 ,
                 col('vv') / 1000 ,
                 col('pres') / 1000,
                 coalesce( col('rr3'),
                           col('rr24')/8,
                           col('rr12')/4,
                           col('rr6')/2,
                           col('rr1')*3  ) )\
             .toDF('id','annee','mois','jour','mois_jour','temperature',
                   'humidite','visibilite','pression','precipitations')\
             .cache()

### orderBy

In [19]:
meteo.where('id < 8000')\
      .select ('id','annee','mois','jour','temperature')\
      .orderBy( 'id','annee','mois','jour','temperature',
                ascending=[1,0,0,0,1])\
      .show(5)

[Stage 12:===========================================>              (3 + 1) / 4]

+----+-----+----+----+-----------+
|  id|annee|mois|jour|temperature|
+----+-----+----+----+-----------+
|7005| 2025|  12|  31|       -0.9|
|7005| 2025|  12|  31|       -0.9|
|7005| 2025|  12|  31|        0.2|
|7005| 2025|  12|  31|        1.1|
|7005| 2025|  12|  31|        2.5|
+----+-----+----+----+-----------+
only showing top 5 rows


In [20]:
meteo.where('id < 8000')\
     .select('annee','mois_jour',
             'temperature')\
     .describe().show(20)

[Stage 13:==============>                                           (1 + 2) / 4]

+-------+------------------+-----------------+------------------+
|summary|             annee|        mois_jour|       temperature|
+-------+------------------+-----------------+------------------+
|  count|            364194|           364194|            354933|
|   mean| 2023.998388221662|668.2302564018079|13.521904134019705|
| stddev|0.8149864536101716|345.3289387380613| 7.255302188633708|
|    min|              2023|             0101|             -12.1|
|    max|              2025|             1231|              42.3|
+-------+------------------+-----------------+------------------+



In [21]:
meteo.count(), meteo.sample(withReplacement=True, fraction=0.01, seed=123456).count()

(525327, 5394)

In [22]:
meteo.sample(True,1/100).count()

5185

In [23]:
meteo.sample(1/100).count()

5236

----------------------------------------
Exception occurred during processing of request from ('127.0.0.1', 33360)
Traceback (most recent call last):
  File "/usr/lib/python3.10/socketserver.py", line 316, in _handle_request_noblock
    self.process_request(request, client_address)
  File "/usr/lib/python3.10/socketserver.py", line 347, in process_request
    self.finish_request(request, client_address)
  File "/usr/lib/python3.10/socketserver.py", line 360, in finish_request
    self.RequestHandlerClass(request, client_address, self)
  File "/usr/lib/python3.10/socketserver.py", line 747, in __init__
    self.handle()
  File "/opt/spark/python/pyspark/accumulators.py", line 299, in handle
    poll(accum_updates)
  File "/opt/spark/python/pyspark/accumulators.py", line 271, in poll
    if self.rfile in r and func():
  File "/opt/spark/python/pyspark/accumulators.py", line 275, in accum_updates
    num_updates = read_int(self.rfile)
  File "/opt/spark/python/pyspark/serializers.py", lin